In [ ]:
!pip install -q tavily-python pymupdf sentence-transformers
!pip install -q "pymilvus[milvus-lite]"
!pip install -q google-genai pandas

In [ ]:
from getpass import getpass

# Enter API keys securely; input is not displayed in the notebook.
TAVILY_API_KEY = getpass("Enter Tavily API key: ").strip()
GEMINI_API_KEY = getpass("Enter Gemini API key: ").strip()

if not TAVILY_API_KEY or not GEMINI_API_KEY:
    raise ValueError("Both Tavily and Gemini API keys are required.")

print("API keys configured successfully.")


In [ ]:

import requests
import fitz

from tavily import TavilyClient

# Connect to Tavily
tavily = TavilyClient(api_key=TAVILY_API_KEY)

company = "Microsoft"

search_topics = {
    "financial_report": {
        "topic": "annual financial report revenue",
        "domains": ["microsoft.com", "sec.gov"]
    },
    "blog": {
        "topic": "official technology business blog",
        "domains": ["blogs.microsoft.com"]
    },
    "press_release": {
        "topic": "official press releases announcements",
        "domains": ["news.microsoft.com"]
    }
}

documents = []
seen_urls = set()

# Search and collect documents
for category, details in search_topics.items():

    results = tavily.search(
        query=f"{company} {details['topic']}",
        search_depth="advanced",
        max_results=5,
        include_domains=details["domains"]
    )

    for result in results["results"]:

        url = result["url"]

        if url in seen_urls:
            continue

        seen_urls.add(url)

        try:

            if url.lower().split("?")[0].endswith(".pdf"):

                response = requests.get(url, timeout=60)
                response.raise_for_status()

                pdf = fitz.open(
                    stream=response.content,
                    filetype="pdf"
                )

                for page_number, page in enumerate(pdf):

                    text = page.get_text()

                    if text.strip():

                        documents.append({
                            "title": result["title"],
                            "text": text,
                            "source": url,
                            "category": category,
                            "page": page_number + 1
                        })

                pdf.close()

            else:

                extracted = tavily.extract(
                    urls=[url],
                    format="markdown"
                )

                for item in extracted["results"]:

                    if item.get("raw_content"):

                        documents.append({
                            "title": result["title"],
                            "text": item["raw_content"],
                            "source": item["url"],
                            "category": category,
                            "page": None
                        })

        except Exception as error:

            print("Extraction failed:", url, error)

print("Total documents collected:", len(documents))

for document in documents[:5]:

    print("\nTitle:", document["title"])
    print("Category:", document["category"])
    print("Source:", document["source"])


In [ ]:

import re

def clean_text(text):

    text = re.sub(r"\s+", " ", text)

    return text.strip()


def create_chunks(documents):

    chunks = []

    chunk_size = 1000
    overlap = 150

    for document in documents:

        text = clean_text(document["text"])

        start = 0

        while start < len(text):

            end = start + chunk_size

            chunk_text = text[start:end]

            if len(chunk_text.strip()) > 50:

                chunks.append({
                    "text": chunk_text,
                    "title": document["title"],
                    "source": document["source"],
                    "category": document["category"],
                    "page": document["page"]
                })

            start = end - overlap

    return chunks


chunks = create_chunks(documents)

print("Total chunks created:", len(chunks))

for chunk in chunks[:3]:

    print("\nSource:", chunk["title"])
    print("Category:", chunk["category"])
    print("Text:", chunk["text"][:200])


In [ ]:

from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texts = [
    chunk["text"] for chunk in chunks
]

# Generate embeddings
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Total embeddings:", len(embeddings))

print("Embedding dimensions:", embeddings.shape)

print("First embedding:", embeddings[0])


In [ ]:

from pymilvus import MilvusClient

client = MilvusClient(
    "business_analysis.db"
)

collection_name = "business_documents"

if client.has_collection(collection_name):

    client.drop_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    dimension=384,
    metric_type="COSINE"
)

data = []

for i, chunk in enumerate(chunks):

    record = {
        "id": i,
        "vector": embeddings[i].tolist(),
        "text": chunk["text"],
        "title": chunk["title"],
        "source": chunk["source"],
        "category": chunk["category"],
        "page": chunk["page"] or 0
    }

    data.append(record)

# Insert data into Milvus
if data:

    client.insert(
        collection_name=collection_name,
        data=data
    )

print("Successfully stored embeddings in Milvus!")

print("Total records stored:", len(data))


In [ ]:

def retrieve_documents(question):

    # Convert analyst question into embedding
    question_embedding = model.encode(
        question,
        normalize_embeddings=True
    )

    # Search relevant information in Milvus
    results = client.search(
        collection_name=collection_name,

        data=[
            question_embedding.tolist()
        ],

        limit=5,

        output_fields=[
            "text",
            "title",
            "source",
            "category",
            "page"
        ]
    )

    retrieved_chunks = []

    for result in results[0]:

        chunk = result["entity"]

        chunk["similarity_score"] = result["distance"]

        retrieved_chunks.append(chunk)

    print("Retrieved chunks:", len(retrieved_chunks))

    return retrieved_chunks


# Test retrieval
question = "What was Microsoft's annual revenue?"

retrieved_chunks = retrieve_documents(question)

for chunk in retrieved_chunks:

    print("\nTitle:", chunk["title"])
    print("Category:", chunk["category"])
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print("Similarity:", chunk["similarity_score"])
    print("Content:", chunk["text"][:200])


In [ ]:

from google import genai
import re

# Connect to Gemini
gemini = genai.Client(
    api_key=GEMINI_API_KEY
)

GEMINI_MODEL = "gemini-2.5-flash"


def generate_answer(question, retrieved_chunks):

    if not retrieved_chunks:

        return "No relevant information was found.", []

    context = ""

    for i, chunk in enumerate(retrieved_chunks):

        context += f"""
        [S{i + 1}]

        Title: {chunk['title']}
        Category: {chunk['category']}
        Source URL: {chunk['source']}
        Page: {chunk['page']}

        Content:
        {chunk['text']}

        """

    prompt = f"""
    You are a business analysis assistant.

    Answer the analyst's question using only
    the retrieved information below.

    Instructions:

    1. Do not invent financial figures.
    2. Do not use outside knowledge.
    3. Preserve financial years and reporting periods.
    4. Cite sources using [S1], [S2], etc.
    5. Cite only sources that directly support your answer.
    6. If the information is unavailable, clearly say so.
    7. Distinguish facts from conflicting information.

    Retrieved Context:
    {context}

    Analyst Question:
    {question}

    Provide a clear and accurate answer.
    """

    response = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt
    )

    answer = response.text

    # Find source references used in the answer
    cited_ids = set(
        int(value)
        for value in re.findall(r"\[S(\d+)\]", answer)
    )

    # Display only references actually cited by the LLM
    sources = []

    for source_id in sorted(cited_ids):

        if 1 <= source_id <= len(retrieved_chunks):

            chunk = retrieved_chunks[source_id - 1]

            sources.append({
                "title": chunk["title"],
                "url": chunk["source"],
                "category": chunk["category"],
                "page": chunk["page"]
            })

    return answer, sources


# Ask the analyst for a question
question = input("Enter your business question: ")

# Retrieve relevant documents
retrieved_chunks = retrieve_documents(question)

# Generate the final answer
answer, sources = generate_answer(
    question,
    retrieved_chunks
)

print("\nFINAL ANSWER:\n")
print(answer)

print("\nSUPPORTING SOURCES:\n")

for source in sources:

    print("Title:", source["title"])
    print("Category:", source["category"])
    print("URL:", source["url"])
    print("Page:", source["page"])
    print("--------------------------------")
